In [1]:
import requests
import pandas as pd
from tqdm import tqdm  # progress bar
from datetime import datetime
import os


In [14]:
def get_historical_weather(lat, lon, start_date, end_date):
    """
    Fetch daily historical weather data from Open-Meteo.

    Parameters:
    ----------
    lat : float
        Latitude
    lon : float
        Longitude
    start_date : str (YYYY-MM-DD)
    end_date : str (YYYY-MM-DD)

    Returns:
    -------
    pd.DataFrame with columns:
        - date
        - temperature
        - precipitation
        - weathercode
        - windspeed
    """

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "weathercode",
            "windspeed_10m_max"
        ],
        "timezone": "auto"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    # Extract daily data
    daily = data.get("daily", {})

    df = pd.DataFrame({
        "date": daily.get("time", []),
        "temperature": daily.get("temperature_2m_mean", []),
        "precipitation": daily.get("precipitation_sum", []),
        "weathercode": daily.get("weathercode", []),
        "windspeed": daily.get("windspeed_10m_max", [])
    })

    return df

In [15]:
def get_forecast_daily(lat, lon, future_days=30):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "weathercode",
            "windspeed_10m_max"
        ],
        "forecast_days": future_days,
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    daily = data.get("daily", {})
    df = pd.DataFrame({
        "date": daily.get("time", []),
        "latitude": lat,
        "longitude": lon,
        "temperature": daily.get("temperature_2m_mean", []),
        "precipitation": daily.get("precipitation_sum", []),
        "weathercode": daily.get("weathercode", []),
        "windspeed": daily.get("windspeed_10m_max", [])
    })
    return df

In [16]:
def get_forecast_hourly(lat, lon):
    """
    Fetch hourly weather forecast (next ~7 days).

    Parameters:
    ----------
    lat : float
    lon : float

    Returns:
    -------
    pd.DataFrame with:
        - datetime
        - temperature
        - precipitation
        - weathercode
        - windspeed
    """

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": [
            "temperature_2m",
            "precipitation",
            "weathercode",
            "windspeed_10m"
        ],
        "forecast_days": 7,   # max reliable hourly range
        "timezone": "auto"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    hourly = data.get("hourly", {})

    df = pd.DataFrame({
        "datetime": hourly.get("time", []),
        "temperature": hourly.get("temperature_2m", []),
        "precipitation": hourly.get("precipitation", []),
        "weathercode": hourly.get("weathercode", []),
        "windspeed": hourly.get("windspeed_10m", [])
    })

    return df

In [17]:
df = pd.read_csv("/home/asad/Desktop/footfall_explorer/data/France/processed_data/sell_out_merged.csv")

In [18]:
def get_historical_weather(lat, lon, start_date, end_date):
    import requests
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "weathercode",
            "windspeed_10m_max"
        ],
        "timezone": "auto"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    daily = data.get("daily", {})

    df = pd.DataFrame({
        "date": daily.get("time", []),
        "temperature": daily.get("temperature_2m_mean", []),
        "precipitation": daily.get("precipitation_sum", []),
        "weathercode": daily.get("weathercode", []),
        "windspeed": daily.get("windspeed_10m_max", [])
    })

    return df



In [19]:
# --- Main function to fetch & save all weather ---
def fetch_and_save_weather(df, start_date="2026-01-01", end_date="2026-04-08", save_path="all_weather.csv"):
    """
    df: Original dataset with ['date','customer_code','customer_name','latitude','longitude']
    start_date, end_date: Historical date range
    save_path: CSV path to save the full weather data
    """

    # 1. Ensure date is datetime
    df['date'] = pd.to_datetime(df['date'])

    # 2. Unique coordinates
    unique_coords = df[['latitude','longitude']].drop_duplicates()

    all_weather = []

    # 3. Fetch weather for each unique lat/lon
    for idx, row in tqdm(unique_coords.iterrows(), total=len(unique_coords), desc="Fetching weather"):
        lat, lon = row['latitude'], row['longitude']
        try:
            weather_df = get_historical_weather(lat, lon, start_date, end_date)
            weather_df['latitude'] = lat
            weather_df['longitude'] = lon
            all_weather.append(weather_df)
        except Exception as e:
            print(f"Failed for {lat}, {lon}: {e}")

    # 4. Concatenate all weather
    weather_full = pd.concat(all_weather, ignore_index=True)

    # 5. Save to single CSV
    weather_full.to_csv(save_path, index=False)
    print(f"All weather data saved to {save_path} ({weather_full.shape[0]} rows)")

    return weather_full

In [20]:
def merge_weather_with_data(df, weather_full):
    """
    Merge original data with full weather on ['date','latitude','longitude']
    """
    merged_df = pd.merge(
        df,
        weather_full,
        on=['date','latitude','longitude'],
        how='left'
    )
    return merged_df


In [22]:
weather_full = fetch_and_save_weather(df, start_date="2026-01-01", end_date="2026-04-08", save_path="/home/asad/Desktop/footfall_explorer/data/France/processed_data/weather_till_march2.csv")


Fetching weather:   0%|          | 0/571 [00:00<?, ?it/s]

Fetching weather: 100%|█████████▉| 570/571 [07:34<00:00,  1.47it/s]

Failed for nan, nan: 400 Client Error: Bad Request for url: https://archive-api.open-meteo.com/v1/archive?latitude=nan&longitude=nan&start_date=2026-01-01&end_date=2026-04-08&daily=temperature_2m_mean&daily=precipitation_sum&daily=weathercode&daily=windspeed_10m_max&timezone=auto


Fetching weather: 100%|██████████| 571/571 [07:35<00:00,  1.25it/s]


All weather data saved to /home/asad/Desktop/footfall_explorer/data/France/processed_data/weather_till_march2.csv (55860 rows)


In [36]:
# Convert weather_full date to datetime for merging
weather_full['date'] = pd.to_datetime(weather_full['date'])

In [38]:
# Merge weather with your dataset (optional)
merged_df = merge_weather_with_data(df, weather_full)

# Save merged data
merged_df.to_csv("/home/asad/Desktop/footfall_explorer/data/France/processed_data/sell_out_with_weather.csv", index=False)

print("Merged data saved successfully!")

Merged data saved successfully!


### GET FORECAST 

In [5]:
def make_wide_weather(df_weather):
    df_weather['date'] = pd.to_datetime(df_weather['date'])
    features = ['temperature','precipitation','weathercode','windspeed']
    wide_dfs = []
    for feature in features:
        temp = df_weather.pivot(index=['latitude','longitude'], columns='date', values=feature)
        temp.columns = [f"{feature}_{d.strftime('%Y-%m-%d')}" for d in temp.columns]
        wide_dfs.append(temp)
    wide_df = pd.concat(wide_dfs, axis=1).reset_index()
    return wide_df

In [10]:
def fetch_forecast_with_external_set(unique_coords, processed_coords, forecast_days=16, forecast_file="forecast_30days.csv"):
    """
    Fetch 16-day forecast for each lat/lon.
    Uses an external set of processed coordinates.
    
    unique_coords: DataFrame with ['latitude','longitude']
    processed_coords: set() of already fetched (lat, lon)
    forecast_days: number of days ahead to fetch
    forecast_file: path to store/load forecasts
    """
    # Load existing forecast if available
    if os.path.exists(forecast_file):
        forecast_full = pd.read_csv(forecast_file)
        print(f"Loaded existing forecast file: {forecast_file} ({forecast_full.shape[0]} rows)")
    else:
        forecast_full = pd.DataFrame()

    # Iterate over unique_coords
    for idx, row in tqdm(unique_coords.iterrows(), total=len(unique_coords), desc="Fetching forecast"):
        lat, lon = row['latitude'], row['longitude']
        # Skip if already fetched
        if (lat, lon) in processed_coords:
            continue
        try:
            # print(f"Fetching forecast for {lat},{lon}")
            forecast_df = get_forecast_daily(lat, lon, future_days=forecast_days)
            # Append to full forecast
            forecast_full = pd.concat([forecast_full, forecast_df], ignore_index=True)
            # Save immediately
            forecast_full.to_csv(forecast_file, index=False)
            # Add to external processed set
            processed_coords.add((lat, lon))
        except Exception as e:
            print(f"Forecast failed for {lat},{lon}: {e}")

    print(f"Forecast fetching complete. Total rows: {forecast_full.shape[0]}")
    return forecast_full

In [11]:
unique_coords = df[['latitude','longitude']].drop_duplicates()
# Drop lat long with 0 values (if any)
unique_coords = unique_coords[(unique_coords['latitude'] != 0) & (unique_coords['longitude'] != 0)]
processed_coords = set()

In [12]:
len(unique_coords)
processed_coords

set()

In [13]:
forecast_30days = fetch_forecast_with_external_set(
    unique_coords=unique_coords,
    processed_coords=processed_coords,
    forecast_days=16,
    forecast_file="/home/asad/Desktop/footfall_explorer/data/France/processed_data/forecast_30days2.csv"
)



Fetching forecast: 100%|█████████▉| 569/570 [09:36<00:00,  1.20it/s]

Forecast failed for nan,nan: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=nan&longitude=nan&daily=temperature_2m_mean&daily=precipitation_sum&daily=weathercode&daily=windspeed_10m_max&forecast_days=16&timezone=auto


Fetching forecast: 100%|██████████| 570/570 [09:37<00:00,  1.01s/it]

Forecast fetching complete. Total rows: 9104


## CONCATINATING BOTH DATASETS. 

In [23]:
def build_wide_weather_from_files(
    weather_file,
    forecast_file
):
    """
    Combine historical + forecast weather and convert to wide format.

    Parameters:
    ----------
    weather_file : str
        Path to historical weather CSV
    forecast_file : str
        Path to forecast weather CSV

    Returns:
    -------
    wide_df : pd.DataFrame
        One row per (lat, lon), columns for each day's weather features
    """

    # 1. Load both files
    weather_df = pd.read_csv(weather_file)
    forecast_df = pd.read_csv(forecast_file)

    # 2. Combine
    full_df = pd.concat([weather_df, forecast_df], ignore_index=True)

    # 3. Convert date
    full_df['date'] = pd.to_datetime(full_df['date'])

    # 4. Sort (important)
    full_df = full_df.sort_values(['latitude','longitude','date'])

    # 5. Features to pivot
    features = ['temperature','precipitation','weathercode','windspeed']

    wide_dfs = []

    # 6. Pivot each feature
    for feature in features:
        temp = full_df.pivot(
            index=['latitude','longitude'],
            columns='date',
            values=feature
        )

        # Rename columns
        temp.columns = [
            f"{feature}_{d.strftime('%Y-%m-%d')}"
            for d in temp.columns
        ]

        wide_dfs.append(temp)

    # 7. Combine all features
    wide_df = pd.concat(wide_dfs, axis=1).reset_index()

    return wide_df

In [25]:
historical_weather_path = "/home/asad/Desktop/footfall_explorer/data/France/processed_data/weather_till_march2.csv"
forecast_30days_path = "/home/asad/Desktop/footfall_explorer/data/France/processed_data/forecast_30days2.csv"

In [26]:
wide_weather = build_wide_weather_from_files(
    weather_file=historical_weather_path,
    forecast_file=forecast_30days_path
)

In [27]:
wide_weather.to_csv("/home/asad/Desktop/footfall_explorer/data/France/processed_data/wide_weather.csv", index=False)